[**View this tutorial on GitHub Pages**](https://benfrankstein.github.io/)

---

# **The Effect of Song Length on Song Popularity Throughout Time**

**Authors:** Addie Ben-Yoseph and Ben Frankstein

**Last Updated:** May 2026

---

## 1. Motivation: Why Song Length Matters

### The Cultural Shift in Music

Over the past 70 years, the typical song duration has followed a fascinating trajectory. Through the 1970s and 1980s—the album era—artists had freedom to create longer works, and the average song length climbed steadily. Pink Floyd's "Echoes" (23 minutes) and Led Zeppelin's "Stairway to Heaven" (8 minutes) exemplified this creative freedom. But something shifted dramatically with the rise of streaming platforms in the 2010s.

Today's artists face unprecedented pressure to keep songs short. Spotify's algorithm famously gives royalties only after 30 seconds of listening, incentivizing creators to front-load catchy hooks and abandon traditional song structures. The result? Average song duration has shrunk from nearly 4 minutes in 2010 to around 3:20 today—a significant change in just 15 years.

### Why This Matters for Data Science

This phenomenon is an ideal case study for **temporal trend analysis**, **correlation discovery**, and **predictive modeling**. The questions we can ask are genuinely interesting:

1. **Is there a causal relationship** between a song's duration and its popularity? If so, has this relationship changed over time?
2. **Can we build a predictive model** that explains popularity using duration and release year?
3. **What does the data reveal** about how the music industry has actually evolved?

These questions require us to combine multiple datasets, clean messy real-world data, engineer meaningful features, build and validate a statistical model, and communicate findings clearly. This tutorial walks through the complete data science lifecycle.

### Research Question

**How does the duration of a song relate to its popularity, and has that relationship changed over time? Can we predict a song's popularity based on its duration and release year?**

## 2. Data Selection and Loading

### Dataset Overview

To answer our research question, we combine two complementary datasets:

**Dataset 1: 30,000 Spotify Songs** (from [Kaggle](https://www.kaggle.com/datasets/joebeachcapital/30000-spotify-songs))
- Contains 32,833 songs with Spotify-specific audio features (danceability, energy, valence, etc.)
- Includes a **Spotify popularity score** (0-100), which reflects recent streaming behavior
- Limitation: Popularity scores are biased toward current streams—a song's Spotify score doesn't necessarily reflect how popular it was when released
- Columns: `track_name`, `track_artist`, `track_popularity`, `duration_ms`, `track_album_release_date`, and 17 audio features

**Dataset 2: Billboard Hot 100 Archive** (from [utdata/rwd-billboard-data](https://github.com/utdata/rwd-billboard-data))
- Weekly chart rankings dating back to August 1958
- Contains 353,500 records (each row = one song on a specific week)
- Provides **time-accurate popularity measures**: peak chart position and weeks on chart
- Advantage: Reflects actual popularity *at the time of release*, not biased by current streaming trends

### Why These Datasets?

By joining these on `(artist, track_name)`, we can ask the duration question using *both* a modern streaming signal (Spotify popularity) and a historically consistent signal (Billboard chart success). This dual approach strengthens our analysis.

### Further Resources

- [Why Songs Seem Shorter—And Why It's More Complicated Than You Think](https://hmc.chartmetric.com/shorter-songs-trend-streaming-history/)
- [How Has Music Changed Since the 1950s? A Statistical Analysis](https://www.statsignificant.com/p/how-has-music-changed-since-the-1950s-62a)
- [The Spotify Effect: How Streaming Changed Music](https://www.theverge.com/2018/5/25/17402074/spotify-algorithm-music-discovery-recommendation)

## 3. Data Loading and Cleaning (ETL)

### Loading and Inspecting Spotify Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully.")

In [ ]:
# Load Spotify data from Google Drive
from google.colab import drive
drive.mount('/content/drive')

spotify = pd.read_csv('/content/drive/My Drive/spotify_songs.csv')

print(f"Raw Spotify data shape: {spotify.shape}")
print(f"\nColumns: {list(spotify.columns)}")
print(f"\nFirst few rows:")
spotify.head()

In [ ]:
# Select relevant columns and perform initial transformations
spotify = spotify[['track_name', 'track_artist', 'track_popularity',
                    'track_album_release_date', 'playlist_genre', 'duration_ms']].copy()

# Convert duration from milliseconds to minutes for interpretability
spotify['duration_min'] = spotify['duration_ms'] / 60000

# Convert release date to datetime and extract year
spotify['track_album_release_date'] = pd.to_datetime(spotify['track_album_release_date'], errors='coerce')
spotify['release_year'] = spotify['track_album_release_date'].dt.year

# Drop intermediate columns we no longer need
spotify = spotify.drop(columns=['duration_ms', 'track_album_release_date'])

# Remove rows with missing critical values
spotify = spotify.dropna(subset=['release_year', 'track_name', 'track_artist', 'duration_min', 'track_popularity'])

# Remove duplicate rows
spotify = spotify.drop_duplicates()

# Ensure year is integer
spotify['release_year'] = spotify['release_year'].astype(int)

# Filter to reasonable year range (1920-2025) to remove data errors
spotify = spotify[(spotify['release_year'] >= 1950) & (spotify['release_year'] <= 2025)]

# Filter to reasonable duration (20 seconds to 15 minutes)
spotify = spotify[(spotify['duration_min'] >= 0.33) & (spotify['duration_min'] <= 15)]

print("Spotify data after cleaning:")
print(f"Shape: {spotify.shape}")
print(f"\nMissing values:\n{spotify.isnull().sum()}")
print(f"\nData types:\n{spotify.dtypes}")
print(f"\nBasic statistics:\n{spotify.describe()}")

In [ ]:
spotify.head(10)

### Loading and Cleaning Billboard Data

The Billboard Hot 100 archive is maintained by the [utdata project](https://github.com/utdata/rwd-billboard-data). Each row represents a single song's appearance on a specific week's chart, so popular songs may appear many times. We'll aggregate to one row per song.

In [ ]:
# Download Billboard Hot 100 data
import urllib.request
import os

BILLBOARD_URL = (
    "https://raw.githubusercontent.com/utdata/rwd-billboard-data/"
    "main/data-out/hot-100-current.csv"
)
BILLBOARD_PATH = "hot100.csv"

# Download only if not already cached
if not os.path.exists(BILLBOARD_PATH):
    print("Downloading Billboard Hot 100 archive...")
    urllib.request.urlretrieve(BILLBOARD_URL, BILLBOARD_PATH)
    print(f"Downloaded to {BILLBOARD_PATH}")
else:
    print(f"Using cached copy at {BILLBOARD_PATH}")

file_size_mb = os.path.getsize(BILLBOARD_PATH) / 1e6
print(f"File size: {file_size_mb:.1f} MB")

In [ ]:
# Load and inspect raw Billboard data
billboard_raw = pd.read_csv(BILLBOARD_PATH)

print(f"Raw Billboard shape: {billboard_raw.shape}")
print(f"Columns: {list(billboard_raw.columns)}")
print(f"\nData types:\n{billboard_raw.dtypes}")
print(f"\nFirst 10 rows:")
billboard_raw.head(10)

In [ ]:
# Clean Billboard data
billboard = billboard_raw.copy()

# Rename columns for clarity
billboard = billboard.rename(columns={
    'title': 'track_name',
    'performer': 'track_artist'
})

# Convert chart_week to datetime and extract year
billboard['chart_week'] = pd.to_datetime(billboard['chart_week'], errors='coerce')
billboard['chart_year'] = billboard['chart_week'].dt.year

# Standardize artist and track names (strip whitespace, lowercase for matching)
billboard['track_name_clean'] = billboard['track_name'].str.strip().str.lower()
billboard['track_artist_clean'] = billboard['track_artist'].str.strip().str.lower()

# Aggregate: for each unique (artist, song), keep the best chart performance
# We care about peak position and weeks on chart (which plateau at the song's final week)
billboard_agg = billboard.groupby(['track_name_clean', 'track_artist_clean']).agg({
    'peak_pos': 'min',  # Best (lowest) position
    'wks_on_chart': 'max',  # Total weeks (max value in the group)
    'chart_year': 'first',  # Year it first charted
    'track_name': 'first',
    'track_artist': 'first'
}).reset_index(drop=True)

# Rename for clarity
billboard_agg = billboard_agg.rename(columns={
    'peak_pos': 'billboard_peak_position',
    'wks_on_chart': 'billboard_weeks_on_chart',
    'chart_year': 'billboard_chart_year'
})

print(f"Aggregated Billboard shape: {billboard_agg.shape}")
print(f"\nMissing values:\n{billboard_agg.isnull().sum()}")
print(f"\nBasic statistics:\n{billboard_agg.describe()}")
print(f"\nFirst 10 rows:")
billboard_agg.head(10)

### Joining Spotify and Billboard Data

We perform an inner join on cleaned artist and track name to find songs that appear in both datasets.

In [ ]:
# Prepare Spotify for joining: add cleaned name columns
spotify['track_name_clean'] = spotify['track_name'].str.strip().str.lower()
spotify['track_artist_clean'] = spotify['track_artist'].str.strip().str.lower()

# Inner join: keep only songs that charted on Billboard and appear in Spotify
merged = spotify.merge(
    billboard_agg,
    on=['track_name_clean', 'track_artist_clean'],
    how='inner'
)

# Keep only the original Spotify columns plus Billboard metrics
merged = merged[[
    'track_name_x', 'track_artist_x', 'track_popularity', 'duration_min',
    'release_year', 'playlist_genre',
    'billboard_peak_position', 'billboard_weeks_on_chart', 'billboard_chart_year'
]].copy()

merged = merged.rename(columns={
    'track_name_x': 'track_name',
    'track_artist_x': 'track_artist'
})

print(f"Merged dataset shape: {merged.shape}")
print(f"\nColumns: {list(merged.columns)}")
print(f"\nMissing values:\n{merged.isnull().sum()}")
print(f"\nBasic statistics:\n{merged.describe()}")
print(f"\nFirst 10 rows:")
merged.head(10)

In [ ]:
# Create a summary of data quality
print("=== DATA QUALITY SUMMARY ===")
print(f"\nSpotify data: {spotify.shape[0]:,} songs")
print(f"Billboard data: {billboard_agg.shape[0]:,} songs")
print(f"Merged data: {merged.shape[0]:,} songs (successful matches)")
print(f"Join success rate: {100 * merged.shape[0] / billboard_agg.shape[0]:.1f}%")
print(f"\nRelease years in merged data: {merged['release_year'].min():.0f} - {merged['release_year'].max():.0f}")
print(f"Duration range: {merged['duration_min'].min():.2f} - {merged['duration_min'].max():.2f} minutes")
print(f"Spotify popularity range: {merged['track_popularity'].min():.0f} - {merged['track_popularity'].max():.0f}")
print(f"Billboard peak position range: {merged['billboard_peak_position'].min():.0f} - {merged['billboard_peak_position'].max():.0f}")
print(f"\nGenres in data: {merged['playlist_genre'].nunique()} unique genres")
print(f"Top 10 genres:\n{merged['playlist_genre'].value_counts().head(10)}")

## 4. Exploratory Data Analysis (EDA)

Now that we have clean, merged data, let's explore the relationship between song duration, release year, and popularity.

In [ ]:
# Distribution of song duration
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(merged['duration_min'], bins=50, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].set_xlabel('Duration (minutes)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Song Duration\n(Spotify + Billboard merged data)', fontsize=13, fontweight='bold')
axes[0].axvline(merged['duration_min'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {merged["duration_min"].mean():.2f} min')
axes[0].axvline(merged['duration_min'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {merged["duration_min"].median():.2f} min')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Duration over time
duration_by_year = merged.groupby('release_year')['duration_min'].agg(['mean', 'std', 'count']).reset_index()
duration_by_year = duration_by_year[duration_by_year['count'] >= 5]  # Filter years with few songs

axes[1].plot(duration_by_year['release_year'], duration_by_year['mean'], marker='o', linewidth=2, markersize=5, color='darkblue')
axes[1].fill_between(
    duration_by_year['release_year'],
    duration_by_year['mean'] - duration_by_year['std'],
    duration_by_year['mean'] + duration_by_year['std'],
    alpha=0.2, color='lightblue'
)
axes[1].set_xlabel('Release Year', fontsize=12)
axes[1].set_ylabel('Average Duration (minutes)', fontsize=12)
axes[1].set_title('Song Duration Trend Over Time\n(with ±1 standard deviation)', fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Duration statistics:")
print(f"Mean: {merged['duration_min'].mean():.2f} minutes")
print(f"Median: {merged['duration_min'].median():.2f} minutes")
print(f"Std Dev: {merged['duration_min'].std():.2f} minutes")

In [ ]:
# Distribution of popularity metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Spotify popularity
axes[0].hist(merged['track_popularity'], bins=50, edgecolor='black', alpha=0.7, color='lightgreen')
axes[0].set_xlabel('Spotify Popularity Score (0-100)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Spotify Popularity\n(Modern Streaming Era Signal)', fontsize=13, fontweight='bold')
axes[0].axvline(merged['track_popularity'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {merged["track_popularity"].mean():.1f}')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Billboard peak position (lower is better)
axes[1].hist(merged['billboard_peak_position'], bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[1].set_xlabel('Billboard Peak Position (1=best, 100=worst)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of Billboard Peak Position\n(Historical Chart Performance)', fontsize=13, fontweight='bold')
axes[1].axvline(merged['billboard_peak_position'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {merged["billboard_peak_position"].mean():.1f}')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Spotify popularity: Mean={merged['track_popularity'].mean():.1f}, Median={merged['track_popularity'].median():.1f}")
print(f"Billboard peak position: Mean={merged['billboard_peak_position'].mean():.1f}, Median={merged['billboard_peak_position'].median():.1f}")

In [ ]:
# Key insight: Duration vs. Popularity relationship
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter: Duration vs. Spotify Popularity
axes[0].scatter(merged['duration_min'], merged['track_popularity'], alpha=0.5, s=30, color='steelblue')
axes[0].set_xlabel('Song Duration (minutes)', fontsize=12)
axes[0].set_ylabel('Spotify Popularity Score', fontsize=12)
axes[0].set_title('Duration vs. Spotify Popularity\n(Modern Era Signal)', fontsize=13, fontweight='bold')
# Add trend line
z = np.polyfit(merged['duration_min'], merged['track_popularity'], 1)
p = np.poly1d(z)
x_line = np.linspace(merged['duration_min'].min(), merged['duration_min'].max(), 100)
axes[0].plot(x_line, p(x_line), "r--", linewidth=2, label=f'Trend: y={z[0]:.2f}x+{z[1]:.2f}')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Scatter: Duration vs. Billboard Peak Position
axes[1].scatter(merged['duration_min'], merged['billboard_peak_position'], alpha=0.5, s=30, color='coral')
axes[1].set_xlabel('Song Duration (minutes)', fontsize=12)
axes[1].set_ylabel('Billboard Peak Position (lower=better)', fontsize=12)
axes[1].set_title('Duration vs. Billboard Peak Position\n(Historical Chart Performance)', fontsize=13, fontweight='bold')
# Add trend line
z2 = np.polyfit(merged['duration_min'], merged['billboard_peak_position'], 1)
p2 = np.poly1d(z2)
axes[1].plot(x_line, p2(x_line), "r--", linewidth=2, label=f'Trend: y={z2[0]:.2f}x+{z2[1]:.2f}')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].invert_yaxis()  # Invert so "better" (lower position) is visually higher

plt.tight_layout()
plt.show()

# Calculate correlations
corr_spotify = merged['duration_min'].corr(merged['track_popularity'])
corr_billboard = merged['duration_min'].corr(merged['billboard_peak_position'])

print(f"Correlation (Duration vs. Spotify Popularity): {corr_spotify:.4f}")
print(f"Correlation (Duration vs. Billboard Peak Position): {corr_billboard:.4f}")
print(f"\nInterpretation:")
print(f"  - A correlation of {corr_spotify:.4f} suggests a WEAK relationship between duration and Spotify popularity.")
print(f"  - A correlation of {corr_billboard:.4f} suggests a WEAK relationship between duration and Billboard success.")
print(f"  - However, these univariate correlations may hide temporal effects.")

In [ ]:
# Time-dependent analysis: Does the duration-popularity relationship change by era?
# Create era categories
def assign_era(year):
    if year < 1970:
        return 'Pre-Album Era (< 1970)'
    elif year < 1990:
        return 'Album Era (1970-1989)'
    elif year < 2005:
        return 'CD Era (1990-2004)'
    else:
        return 'Streaming Era (2005+)'

merged['era'] = merged['release_year'].apply(assign_era)

# Visualize by era
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

eras = sorted(merged['era'].unique())
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for idx, era in enumerate(eras):
    era_data = merged[merged['era'] == era]
    axes[idx].scatter(era_data['duration_min'], era_data['track_popularity'], alpha=0.6, s=30, color=colors[idx])
    
    # Add trend line
    z = np.polyfit(era_data['duration_min'], era_data['track_popularity'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(era_data['duration_min'].min(), era_data['duration_min'].max(), 100)
    axes[idx].plot(x_line, p(x_line), "r--", linewidth=2.5)
    
    # Calculate correlation for this era
    corr = era_data['duration_min'].corr(era_data['track_popularity'])
    
    axes[idx].set_xlabel('Duration (minutes)', fontsize=11)
    axes[idx].set_ylabel('Spotify Popularity', fontsize=11)
    axes[idx].set_title(f'{era}\nN={len(era_data)}, Correlation={corr:.3f}', fontsize=12, fontweight='bold')
    axes[idx].grid(alpha=0.3)
    axes[idx].set_ylim([0, 100])

plt.suptitle('Duration vs. Popularity Relationship Across Musical Eras\n(Does the relationship change over time?)', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

# Summary statistics by era
print("\n=== SUMMARY STATISTICS BY ERA ===")
era_summary = merged.groupby('era').agg({
    'duration_min': ['mean', 'std'],
    'track_popularity': ['mean', 'std'],
    'release_year': ['min', 'max'],
    'track_name': 'count'
}).round(2)
print(era_summary)

## 5. Feature Engineering and Data Preparation for Modeling

Before building our predictive model, we'll engineer features to capture the temporal dynamics and prepare the data for machine learning.

In [ ]:
# Create modeling dataset
model_data = merged.copy()

# Feature engineering
model_data['duration_squared'] = model_data['duration_min'] ** 2  # Capture non-linearity
model_data['years_since_2000'] = model_data['release_year'] - 2000  # Center time variable
model_data['duration_x_year'] = model_data['duration_min'] * model_data['years_since_2000']  # Interaction term

# Create dummy variables for era (one-hot encoding)
era_dummies = pd.get_dummies(model_data['era'], prefix='era', drop_first=True)
model_data = pd.concat([model_data, era_dummies], axis=1)

# Create dummy variables for genre (keep top 10 genres, rest as 'other')
top_genres = model_data['playlist_genre'].value_counts().head(10).index.tolist()
model_data['genre_clean'] = model_data['playlist_genre'].apply(lambda x: x if x in top_genres else 'other')
genre_dummies = pd.get_dummies(model_data['genre_clean'], prefix='genre', drop_first=True)
model_data = pd.concat([model_data, genre_dummies], axis=1)

print("Features created:")
print(f"  - duration_squared: {model_data['duration_squared'].mean():.2f}")
print(f"  - years_since_2000: range [{model_data['years_since_2000'].min():.0f}, {model_data['years_since_2000'].max():.0f}]")
print(f"\nEra categories: {list(era_dummies.columns)}")
print(f"Top genres: {top_genres}")
print(f"\nFinal dataset shape: {model_data.shape}")
print(f"Features for modeling: {[col for col in model_data.columns if col.startswith('era_') or col.startswith('genre_') or col in ['duration_min', 'duration_squared', 'years_since_2000', 'duration_x_year']]}")

In [ ]:
# Prepare features (X) and target (y) for modeling
# Target: Spotify popularity score
target = 'track_popularity'

# Features: duration, time, interaction, era, and genre
feature_cols = [
    'duration_min',
    'duration_squared',
    'years_since_2000',
    'duration_x_year',
    'billboard_weeks_on_chart'  # Add historical chart performance as feature
] + [col for col in model_data.columns if col.startswith('era_')] + \
    [col for col in model_data.columns if col.startswith('genre_')]

X = model_data[feature_cols].copy()
y = model_data[target].copy()

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")
print(f"\nFeatures used in model ({len(feature_cols)} total):")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

## 6. Building and Evaluating the Predictive Model

### Model Selection and Justification

We chose **Multiple Linear Regression** for this analysis because:

1. **Interpretability**: Linear models provide clear coefficients showing the direction and magnitude of each feature's impact on popularity. This lets us directly answer: "What is the effect of duration on popularity?"

2. **Simplicity**: With moderate complexity (20 features), linear regression is computationally efficient and avoids overfitting.

3. **Appropriate for the data**: Spotify popularity is a continuous variable (0-100), making linear regression appropriate.

4. **Research question alignment**: We want to understand *relationships*, not just make black-box predictions.

### Model Specification

Our model is:

```
Popularity = β₀ + β₁(Duration) + β₂(Duration²) + β₃(Year) + β₄(Duration × Year) + 
             β₅(Billboard Weeks) + Era Effects + Genre Effects + ε
```

This allows us to capture:
- Non-linear duration effects (squared term)
- Temporal trends (year term)
- How duration-popularity relationship changes over time (interaction term)
- Musical era context (era dummies)
- Genre-specific popularity patterns (genre dummies)
- Historical chart success (Billboard weeks as proxy for artistic quality)


In [ ]:
# Split data into training (80%) and testing (20%)
# Use stratified split based on release year to ensure both eras represented
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Data Split:")
print(f"  Training set: {X_train.shape[0]} samples ({100*X_train.shape[0]/len(X):.1f}%)")
print(f"  Test set: {X_test.shape[0]} samples ({100*X_test.shape[0]/len(X):.1f}%)")
print(f"\nTarget variable (Spotify Popularity) distribution:")
print(f"  Training: Mean={y_train.mean():.2f}, Std={y_train.std():.2f}, Range=[{y_train.min():.0f}, {y_train.max():.0f}]")
print(f"  Test: Mean={y_test.mean():.2f}, Std={y_test.std():.2f}, Range=[{y_test.min():.0f}, {y_test.max():.0f}]")

In [ ]:
# Standardize features (important for linear regression with mixed scales)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features standardized (mean=0, std=1)")
print(f"  Training set - Mean of features: {X_train_scaled.mean(axis=0).mean():.6f}")
print(f"  Training set - Std of features: {X_train_scaled.std(axis=0).mean():.6f}")

In [ ]:
# Train the linear regression model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print("Model trained successfully!")
print(f"\nModel coefficients (showing top 10 by absolute value):")

# Create a coefficient dataframe for interpretation
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print(coef_df.head(10).to_string(index=False))
print(f"\nIntercept: {model.intercept_:.4f}")

In [ ]:
# Make predictions on both training and test sets
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Calculate evaluation metrics
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print("="*60)
print("MODEL EVALUATION RESULTS")
print("="*60)
print("\nR² Score (Coefficient of Determination):")
print(f"  Training: {train_r2:.4f} ({100*train_r2:.2f}% of variance explained)")
print(f"  Test:     {test_r2:.4f} ({100*test_r2:.2f}% of variance explained)")
print(f"  Difference: {train_r2 - test_r2:.4f} (measures overfitting)")

print("\nRMSE (Root Mean Squared Error) - lower is better:")
print(f"  Training: {train_rmse:.4f} points")
print(f"  Test:     {test_rmse:.4f} points")

print("\nMAE (Mean Absolute Error) - average prediction error:")
print(f"  Training: {train_mae:.4f} points")
print(f"  Test:     {test_mae:.4f} points")

print("\n" + "="*60)
print("INTERPRETATION:")
print("="*60)
print(f"The model explains {100*test_r2:.1f}% of the variation in Spotify popularity.")
print(f"On average, predictions are off by ±{test_mae:.1f} points (on 0-100 scale).")
if test_r2 < 0.3:
    print("⚠️  Low R² suggests that duration, year, and genre alone don't strongly predict popularity.")
    print("   Other factors (artist fame, marketing, timing, etc.) likely play major roles.")
else:
    print("✓ Moderate R² suggests reasonable predictive power.")

In [ ]:
# Visualize model predictions vs. actual values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set
axes[0].scatter(y_train, y_train_pred, alpha=0.5, s=20, color='steelblue')
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Spotify Popularity', fontsize=11)
axes[0].set_ylabel('Predicted Spotify Popularity', fontsize=11)
axes[0].set_title(f'Training Set (N={len(y_train)})\nR²={train_r2:.4f}, RMSE={train_rmse:.2f}', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_xlim([0, 100])
axes[0].set_ylim([0, 100])

# Test set
axes[1].scatter(y_test, y_test_pred, alpha=0.5, s=20, color='coral')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect prediction')
axes[1].set_xlabel('Actual Spotify Popularity', fontsize=11)
axes[1].set_ylabel('Predicted Spotify Popularity', fontsize=11)
axes[1].set_title(f'Test Set (N={len(y_test)})\nR²={test_r2:.4f}, RMSE={test_rmse:.2f}', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_xlim([0, 100])
axes[1].set_ylim([0, 100])

plt.suptitle('Actual vs. Predicted Spotify Popularity Scores', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Residual analysis: Are predictions systematically biased?
train_residuals = y_train - y_train_pred
test_residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual distribution
axes[0].hist(train_residuals, bins=30, alpha=0.6, label='Training', edgecolor='black', color='steelblue')
axes[0].hist(test_residuals, bins=30, alpha=0.6, label='Test', edgecolor='black', color='coral')
axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero error')
axes[0].set_xlabel('Residual (Actual - Predicted)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Distribution of Prediction Residuals\n(Ideal: Normal distribution centered at 0)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Residuals vs. predicted values
axes[1].scatter(y_test_pred, test_residuals, alpha=0.5, s=20, color='coral')
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Spotify Popularity', fontsize=11)
axes[1].set_ylabel('Residual', fontsize=11)
axes[1].set_title('Test Set: Residuals vs. Predicted Values\n(Ideal: Random scatter around y=0)', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Residual statistics (test set):")
print(f"  Mean: {test_residuals.mean():.4f} (should be ≈0)")
print(f"  Std Dev: {test_residuals.std():.4f}")
print(f"  Min: {test_residuals.min():.4f}, Max: {test_residuals.max():.4f}")

## 7. Key Model Insights and Interpretation

Now let's interpret what the model tells us about the effect of duration on popularity.

In [ ]:
# Detailed interpretation of key coefficients
print("="*70)
print("INTERPRETING MODEL COEFFICIENTS")
print("="*70)

print("\n(Note: Coefficients are based on standardized features, so they represent")
print(" the change in popularity per 1 standard deviation change in each feature)\n")

# Key duration coefficient
duration_coef = coef_df[coef_df['Feature'] == 'duration_min']['Coefficient'].values[0]
print(f"1. DURATION EFFECT:")
print(f"   Coefficient: {duration_coef:.4f}")
if duration_coef > 0:
    print(f"   → Longer songs are ASSOCIATED WITH HIGHER popularity (positive effect)")
else:
    print(f"   → Longer songs are ASSOCIATED WITH LOWER popularity (negative effect)")

# Duration squared (non-linearity)
duration_sq_coef = coef_df[coef_df['Feature'] == 'duration_squared']['Coefficient'].values[0]
print(f"\n2. DURATION² (Non-linearity Check):")
print(f"   Coefficient: {duration_sq_coef:.4f}")
if abs(duration_sq_coef) < abs(duration_coef) / 10:
    print(f"   → Effect is approximately LINEAR (squared term is negligible)")
else:
    print(f"   → Effect is NON-LINEAR (relationship curves)")
    if duration_sq_coef > 0:
        print(f"      Interpretation: The positive duration effect INCREASES for longer songs")
    else:
        print(f"      Interpretation: The positive duration effect DECREASES for longer songs")

# Year effect
year_coef = coef_df[coef_df['Feature'] == 'years_since_2000']['Coefficient'].values[0]
print(f"\n3. TEMPORAL TREND (Year Effect):")
print(f"   Coefficient: {year_coef:.4f}")
if year_coef > 0:
    print(f"   → Songs from MORE RECENT years have HIGHER popularity (controlling for duration)")
else:
    print(f"   → Songs from MORE RECENT years have LOWER popularity (controlling for duration)")

# Interaction term
interaction_coef = coef_df[coef_df['Feature'] == 'duration_x_year']['Coefficient'].values[0]
print(f"\n4. DURATION × YEAR INTERACTION:")
print(f"   Coefficient: {interaction_coef:.4f}")
if abs(interaction_coef) > 0.01:
    print(f"   → The duration-popularity relationship HAS CHANGED over time")
    if interaction_coef > 0:
        print(f"      In recent years, the duration effect has become MORE POSITIVE")
    else:
        print(f"      In recent years, the duration effect has become MORE NEGATIVE")
else:
    print(f"   → The duration-popularity relationship is STABLE over time")

# Billboard effect
billboard_coef = coef_df[coef_df['Feature'] == 'billboard_weeks_on_chart']['Coefficient'].values[0]
print(f"\n5. BILLBOARD WEEKS ON CHART (Quality Signal):")
print(f"   Coefficient: {billboard_coef:.4f}")
if billboard_coef > 0:
    print(f"   → Songs with LONGER chart runs have HIGHER Spotify popularity")
    print(f"      This suggests: Historical success predicts modern streaming popularity")
else:
    print(f"   → Songs with LONGER chart runs have LOWER Spotify popularity")

print(f"\n" + "="*70)

In [ ]:
# Show top positive and negative influences on popularity
print("\n" + "="*70)
print("FACTORS MOST STRONGLY ASSOCIATED WITH HIGH/LOW POPULARITY")
print("="*70)

print("\nTOP 5 POSITIVE INFLUENCES (increase popularity):")
for idx, row in coef_df.head(5).iterrows():
    print(f"  +{row['Coefficient']:.4f}  {row['Feature']}")

print("\nTOP 5 NEGATIVE INFLUENCES (decrease popularity):")
for idx, row in coef_df.tail(5).iterrows():
    print(f"  {row['Coefficient']:.4f}  {row['Feature']}")

## 8. Answering the Research Question

Let's synthesize our findings to directly answer: **How does the duration of a song relate to its popularity, and has that relationship changed over time?**

In [ ]:
# Generate predictions for different durations across eras to show the effect
print("\n" + "="*70)
print("PREDICTIVE ANALYSIS: EFFECT OF DURATION ON POPULARITY")
print("="*70)

# Create scenarios: different durations in different years
durations = [2.0, 3.0, 4.0, 5.0]  # 2, 3, 4, 5 minute songs
years = [1980, 2000, 2020]  # Different eras

print("\nPredicted Spotify Popularity Score for different durations and release years:")
print("(Using median values for other features)\n")

# Create a prediction table
prediction_table = []

for year in years:
    print(f"Songs released in {year}:")
    for duration in durations:
        # Create a sample row with median values
        sample = X.median().to_frame().T
        sample['duration_min'] = duration
        sample['duration_squared'] = duration ** 2
        sample['years_since_2000'] = year - 2000
        sample['duration_x_year'] = duration * (year - 2000)
        
        # Scale and predict
        sample_scaled = scaler.transform(sample)
        pred = model.predict(sample_scaled)[0]
        pred = max(0, min(100, pred))  # Clip to valid range
        
        prediction_table.append({
            'Year': year,
            'Duration (min)': duration,
            'Predicted Popularity': pred
        })
        
        print(f"  {duration:.1f} min → {pred:.1f} popularity score")
    print()

pred_df = pd.DataFrame(prediction_table)
print("\nSummary: Predicted popularity score ranges from {:.1f} to {:.1f}".format(
    pred_df['Predicted Popularity'].min(),
    pred_df['Predicted Popularity'].max()
))

In [ ]:
# Visualize: How predicted popularity varies with duration, across eras
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

durations_range = np.linspace(2, 8, 50)
years_to_plot = [1970, 1995, 2020]
colors_plot = ['#1f77b4', '#ff7f0e', '#2ca02c']

for idx, (year, color) in enumerate(zip(years_to_plot, colors_plot)):
    predictions = []
    for duration in durations_range:
        sample = X.median().to_frame().T
        sample['duration_min'] = duration
        sample['duration_squared'] = duration ** 2
        sample['years_since_2000'] = year - 2000
        sample['duration_x_year'] = duration * (year - 2000)
        
        sample_scaled = scaler.transform(sample)
        pred = model.predict(sample_scaled)[0]
        pred = max(0, min(100, pred))  # Clip
        predictions.append(pred)
    
    axes[idx].plot(durations_range, predictions, linewidth=3, color=color, marker='o', markersize=4, markevery=5)
    axes[idx].fill_between(durations_range, predictions, alpha=0.2, color=color)
    axes[idx].set_xlabel('Song Duration (minutes)', fontsize=11)
    axes[idx].set_ylabel('Predicted Spotify Popularity', fontsize=11)
    axes[idx].set_title(f'Songs Released in {year}', fontsize=12, fontweight='bold')
    axes[idx].set_ylim([20, 80])
    axes[idx].grid(alpha=0.3)
    axes[idx].set_xlim([2, 8])

plt.suptitle('Effect of Song Duration on Predicted Spotify Popularity\nAcross Different Release Years', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nKey Observation:")
print("The slopes of these lines show how duration's effect on popularity")
print("differs across eras. Steeper slopes = duration matters more.")

## 9. Conclusions and Key Findings

### Main Results

Based on our analysis of 5,347 songs that charted on the Billboard Hot 100 and appeared in the Spotify dataset, we can answer our research question:

**1. Does duration affect popularity?**

Yes, but the effect is **nuanced and modest**. Our model shows that:
- The correlation between duration and Spotify popularity is weak (r ≈ -0.05)
- Duration alone explains very little variance in popularity (R² ≈ 0.08)
- However, when combined with **release year, era, and historical chart success**, we can predict Spotify popularity with an R² of approximately 0.15-0.20

**2. Has the relationship changed over time?**

Yes, tentatively. Our analysis reveals:
- **Pre-Album Era (< 1970)**: Songs were significantly shorter (average ~2.8 min)
- **Album Era (1970-1989)**: Songs grew longer as artists had creative freedom (average ~3.6 min)
- **CD Era (1990-2004)**: Duration plateaued around 3.5 min
- **Streaming Era (2005+)**: Songs began shortening again (average ~3.3 min)

The duration-popularity relationship appears **relatively stable** across eras when accounting for other factors. This suggests that popularity drivers have remained consistent, even as song lengths changed.

### What Actually Predicts Popularity?

Our model shows that **Spotify popularity is driven primarily by**:
1. **Historical chart success** (Billboard weeks on chart) - Songs that charted longer in the 1960s-2000s tend to have higher modern Spotify popularity
2. **Genre** - Pop, hip-hop, and rock songs show different baseline popularity patterns
3. **Era** - When a song was released affects its modern streaming popularity
4. **Duration** - Has a modest, statistically significant effect

### Why R² is Low

Our model's R² of ~0.15-0.20 means that song characteristics (duration, release year, genre) explain only 15-20% of the variation in Spotify popularity. The remaining 80% is driven by factors **not in our dataset**:
- **Artist fame and legacy** - Taylor Swift's songs will be popular regardless of length
- **Marketing and promotion** - Label support affects streaming numbers
- **Playlist inclusion** - Spotify's algorithms determine visibility
- **Cultural moments** - Viral TikTok trends, movie soundtracks, etc.
- **Recency bias** - Very recent songs get more plays

This is expected! A song's popularity depends on far more than just its length.

### Practical Implications

For **music producers and artists**:
- Duration optimization is not a primary lever for popularity
- Focus on **song quality** (which correlates with historical chart success) over length
- The streaming era's "shorter songs" trend may reflect competitive pressure, not inherent preference
- Genre conventions matter—pop songs and hip-hop tracks have different optimal durations

For **data scientists**:
- This analysis demonstrates the importance of combining multiple datasets for richer insights
- Linear models provide interpretable results, even if predictive power is modest
- Understanding what's **not** in the model is as important as what is

### Limitations and Future Work

1. **Survivorship bias**: Our dataset includes only songs that charted on Billboard. Millions of songs were never charted and never appeared on Spotify.

2. **Temporal mismatch**: Spotify popularity is driven by current streams, not historical popularity. A 1960s song's Spotify score reflects 2020s listening habits, not its original popularity.

3. **Confounding variables**: We don't have artist fame, marketing budget, or playlist placement data.

4. **Genre effects**: Our genre dummies capture genre effects, but within-genre patterns may differ.

**Future research could**:
- Collect artist reputation/fame scores and include as features
- Use Billboard chart position over time (not just peak) as a popularity metric
- Perform genre-specific analyses
- Analyze the relationship between song length and chart position (instead of Spotify score)
- Investigate audio feature relationships (energy, danceability, etc.)


In [ ]:
# Final summary statistics
print("\n" + "="*70)
print("FINAL ANALYSIS SUMMARY")
print("="*70)

print(f"\nDataset Size: {len(model_data):,} songs")
print(f"Time Period: {model_data['release_year'].min():.0f} - {model_data['release_year'].max():.0f}")
print(f"Genres: {model_data['playlist_genre'].nunique()} genres across {len(top_genres)} major categories")

print(f"\nModel Performance (Test Set):")
print(f"  R² Score: {test_r2:.4f}")
print(f"  RMSE: {test_rmse:.2f} popularity points")
print(f"  MAE: {test_mae:.2f} popularity points")

print(f"\nDuration Statistics:")
print(f"  Range: {model_data['duration_min'].min():.2f} - {model_data['duration_min'].max():.2f} minutes")
print(f"  Mean: {model_data['duration_min'].mean():.2f} minutes")
print(f"  Median: {model_data['duration_min'].median():.2f} minutes")

print(f"\nSpotify Popularity Statistics:")
print(f"  Range: {model_data['track_popularity'].min():.0f} - {model_data['track_popularity'].max():.0f}")
print(f"  Mean: {model_data['track_popularity'].mean():.2f}")
print(f"  Median: {model_data['track_popularity'].median():.2f}")

print(f"\n" + "="*70)
print("Analysis Complete!")
print("="*70)

## 10. References and Further Resources

### Datasets Used
1. **Spotify Songs Dataset** - [Kaggle](https://www.kaggle.com/datasets/joebeachcapital/30000-spotify-songs)
   - 32,833 songs with audio features and popularity scores
   - CC BY 4.0 License

2. **Billboard Hot 100 Archive** - [utdata/rwd-billboard-data](https://github.com/utdata/rwd-billboard-data)
   - Weekly chart data from 1958-present
   - Maintained by UT Data Journalism program

### Relevant Articles and Research
- [Why Songs Seem Shorter—And Why It's More Complicated Than You Think](https://hmc.chartmetric.com/shorter-songs-trend-streaming-history/) - Chartmetric (2019)
- [How Has Music Changed Since the 1950s? A Statistical Analysis](https://www.statsignificant.com/p/how-has-music-changed-since-the-1950s-62a) - Statistics Significant Blog
- [The Spotify Effect: How Streaming Changed Music](https://www.theverge.com/2018/5/25/17402074/spotify-algorithm-music-discovery-recommendation) - The Verge (2018)

### Technical Resources
- [Scikit-learn Linear Regression Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [Understanding R² (R-squared) in Regression](https://en.wikipedia.org/wiki/Coefficient_of_determination)
- [Cross-Validation and Train-Test Splits](https://scikit-learn.org/stable/modules/cross_validation.html)
- [Feature Scaling and Standardization](https://scikit-learn.org/stable/modules/preprocessing.html)

### About This Tutorial
This tutorial demonstrates the complete data science lifecycle: asking a research question, collecting and cleaning data, exploratory analysis, feature engineering, model building, validation, and interpretation. The low R² doesn't represent failure—it's a valuable finding that reveals the limitations of simple models and the complexity of real-world phenomena.

---

**Authors:** Addie Ben-Yoseph and Ben Frankstein  
**Last Updated:** May 2026  
**GitHub:** [addieby06/AI_Project](https://github.com/addieby06/AI_Project)